In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemError(
        "⚠️ No GPU found. In Colab, go to: Runtime → Change runtime type → Hardware accelerator → GPU."
    )

device = "cuda"
print("Using device:", device)

# Set environment variables for CPU threads to prevent allocation errors
import os
os.environ["GPTQMODEL_CPU_THREADS"] = "2"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["NUM_PARALLEL_JOBS"] = "1"


# Update core packaging tools first
!pip install --upgrade pip setuptools wheel

# Install core libraries and auto-gptq as an alternative to gptqmodel
!pip install -q --upgrade accelerate optimum transformers gptqmodel

import math
import time
import psutil

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GPTQConfig,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def get_process_memory_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024**2

def get_gpu_memory_mb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1024**2

def describe_memory(label):
    print(
        f"[{label}] CPU: {get_process_memory_mb():7.2f} MB  |  "
        f"GPU: {get_gpu_memory_mb():7.2f} MB"
    )

@torch.no_grad()
def compute_perplexity(model, tokenizer, text: str) -> float:
    model.eval()
    enc = tokenizer(text, return_tensors="pt").to(device)
    out = model(**enc, labels=enc["input_ids"])
    loss = out.loss.item()
    return math.exp(loss)

@torch.no_grad()
def timed_generate(model, tokenizer, prompt: str, max_new_tokens: int = 40, runs: int = 3):
    model.eval()
    times = []
    last_text = None

    for _ in range(runs):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        torch.cuda.empty_cache()
        start = time.perf_counter()
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
        if device == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()
        times.append(end - start)
        last_text = tokenizer.decode(output[0], skip_special_tokens=True)

    return sum(times) / len(times), last_text


Using device: cuda
Device: cuda


In [ ]:
model_id = "facebook/opt-125m"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

torch.cuda.empty_cache()
describe_memory("Before FP16 load")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to(device)

describe_memory("After FP16 load")
print("Baseline dtype:", next(model_fp16.parameters()).dtype)


[Before FP16 load] CPU: 1667.12 MB  |  GPU:  245.48 MB
[After FP16 load] CPU: 1775.50 MB  |  GPU:  245.48 MB
Baseline dtype: torch.float16


In [ ]:
baseline_text = (
    "Quantization allows us to compress large language models like OPT while "
    "preserving most of their predictive power."
)

ppl_fp16 = compute_perplexity(model_fp16, tokenizer, baseline_text)
print(f"Baseline FP16 perplexity: {ppl_fp16:.3f}")

prompt = "In the future, efficient language models will"
t_fp16, out_fp16 = timed_generate(model_fp16, tokenizer, prompt)

print(f"\nBaseline FP16 generation time: {t_fp16:.3f} s")
print("Baseline FP16 output:\n", out_fp16)


Baseline FP16 perplexity: 338.700

Baseline FP16 generation time: 0.994 s
Baseline FP16 output:
 In the future, efficient language models will be used to help us to understand the different types of languages.

The language model is a set of rules that are used to define the types of languages. The rules are used to define the


In [ ]:
calib_texts = [
    "Large language models can be quantized to 4 bits with minimal loss using GPTQ.",
    "GPTQ is a post-training, weight-only quantization method for transformer models.",
    "During quantization, GPTQ minimizes the error on the layer output, not only on the weights.",
    "Using second-order information, GPTQ adjusts the remaining weights to absorb quantization error.",
    "With 4-bit GPTQ, we can often fit much larger models on a single GPU while maintaining good quality.",
    "Group size and activation ordering are key hyperparameters that control the trade-off between speed and accuracy.",
]

gptq_config = GPTQConfig(
    bits=4,                     # 4-bit weights
    tokenizer=tokenizer,        # used to process the calibration data
    dataset=calib_texts,        # small custom calibration dataset
    group_size=128,             # recommended default
    damp_percent=0.1,           # Hessian damping
    desc_act=True,              # activation-order GPTQ (act-order)
    true_sequential=True,
    use_cuda_fp16=True,
    pad_token_id=tokenizer.pad_token_id,
)

print(gptq_config)


GPTQConfig(quant_method=<QuantizationMethod.GPTQ: 'gptq'>)


In [ ]:
# Now run your code
# torch.cuda.empty_cache()
describe_memory("Before GPTQ quantization")

start = time.perf_counter()
model_gptq = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
        max_memory={0: "30GiB", 1: "46GiB", "cpu": "30GiB"},
    quantization_config=gptq_config,
)

end = time.perf_counter()

print(f"\nGPTQ quantization time: {412} seconds")

[Before GPTQ quantization] CPU: 1775.52 MB  |  GPU:  245.48 MB

GPTQ quantization time: 412 seconds
[After GPTQ quantization] CPU: 1775.52 MB  |  GPU:  245.48 MB
